---
title: "Lab 2: Manual Sharding with PostgreSQL (Short Version)"
author: "Dr. Tim Smith"
date: "March 4, 2026"
format:
  html:
    theme: cosmo
    toc: true
    toc-depth: 3
    toc-location: left
    code-fold: false
    code-copy: true
    embed-resources: true
    number-sections: true
    page-layout: full
execute:
  eval: false
---

## Overview

This is the condensed in-class version of the sharding lab. It covers schema creation, reference tables, shard routing, data insertion, single-shard queries, cross-shard queries, and cross-shard aggregation. For schema drift, reference table inconsistency, hash routing, index locality, FK limitations, and dblink coordinator queries, see the **comprehensive lab**.

**Prerequisites:** Start the Docker environment with `cd sharding-lab && docker compose up -d` and wait for all containers to be healthy.

## Architecture

The lab runs three independent PostgreSQL instances on your machine, each representing a regional warehouse shard. Your Python code acts as the **shard router**, deciding which database to query based on the region.

<div style="text-align: center; margin: 1.5em 0;">
<svg viewBox="0 0 760 340" style="max-width: 760px; font-family: sans-serif;">
  <defs>
    <marker id="arr-arch" viewBox="0 0 10 10" refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" fill="#2874a6"/></marker>
  </defs>
  <!-- Application -->
  <rect x="255" y="8" width="250" height="44" rx="8" fill="#d6eaf8" stroke="#2874a6" stroke-width="2"/>
  <text x="380" y="25" text-anchor="middle" font-weight="bold" font-size="13">Python Application</text>
  <text x="380" y="42" text-anchor="middle" font-size="11" fill="#555">(shard router — all queries start here)</text>
  <!-- Arrows -->
  <line x1="310" y1="52" x2="130" y2="105" stroke="#2874a6" stroke-width="2" marker-end="url(#arr-arch)"/>
  <line x1="380" y1="52" x2="380" y2="105" stroke="#2874a6" stroke-width="2" marker-end="url(#arr-arch)"/>
  <line x1="450" y1="52" x2="630" y2="105" stroke="#2874a6" stroke-width="2" marker-end="url(#arr-arch)"/>
  <text x="195" y="75" text-anchor="middle" font-size="10" fill="#2874a6" font-weight="bold">region=east</text>
  <text x="405" y="75" text-anchor="end" font-size="10" fill="#2874a6" font-weight="bold">region=central</text>
  <text x="565" y="75" text-anchor="middle" font-size="10" fill="#2874a6" font-weight="bold">region=west</text>
  <!-- East Shard -->
  <rect x="10" y="110" width="230" height="220" rx="8" fill="#d5f5e3" stroke="#27ae60" stroke-width="2"/>
  <text x="125" y="132" text-anchor="middle" font-weight="bold" font-size="12">East Shard (port 5433)</text>
  <text x="125" y="148" text-anchor="middle" font-size="10" fill="#555">shard1_db — Newark, NJ</text>
  <rect x="25" y="160" width="200" height="28" rx="4" fill="#fff" stroke="#6c3483" stroke-width="1"/>
  <text x="125" y="179" text-anchor="middle" font-size="11">warehouses <tspan fill="#888" font-size="9">(replicated)</tspan></text>
  <rect x="25" y="196" width="200" height="28" rx="4" fill="#fff" stroke="#2874a6" stroke-width="1"/>
  <text x="125" y="215" text-anchor="middle" font-size="11">sensors <tspan fill="#888" font-size="9">(101–108)</tspan></text>
  <rect x="25" y="232" width="200" height="28" rx="4" fill="#fff" stroke="#c0392b" stroke-width="1"/>
  <text x="125" y="251" text-anchor="middle" font-size="11">sensor_readings <tspan fill="#888" font-size="9">(~10K)</tspan></text>
  <text x="125" y="278" text-anchor="middle" font-size="10" fill="#27ae60" font-style="italic">Single-shard queries</text>
  <text x="125" y="292" text-anchor="middle" font-size="10" fill="#27ae60" font-style="italic">are fast — local only</text>
  <!-- Central Shard -->
  <rect x="265" y="110" width="230" height="220" rx="8" fill="#d5f5e3" stroke="#27ae60" stroke-width="2"/>
  <text x="380" y="132" text-anchor="middle" font-weight="bold" font-size="12">Central Shard (port 5434)</text>
  <text x="380" y="148" text-anchor="middle" font-size="10" fill="#555">shard2_db — Dallas, TX</text>
  <rect x="280" y="160" width="200" height="28" rx="4" fill="#fff" stroke="#6c3483" stroke-width="1"/>
  <text x="380" y="179" text-anchor="middle" font-size="11">warehouses <tspan fill="#888" font-size="9">(replicated)</tspan></text>
  <rect x="280" y="196" width="200" height="28" rx="4" fill="#fff" stroke="#2874a6" stroke-width="1"/>
  <text x="380" y="215" text-anchor="middle" font-size="11">sensors <tspan fill="#888" font-size="9">(201–208)</tspan></text>
  <rect x="280" y="232" width="200" height="28" rx="4" fill="#fff" stroke="#c0392b" stroke-width="1"/>
  <text x="380" y="251" text-anchor="middle" font-size="11">sensor_readings <tspan fill="#888" font-size="9">(~10K)</tspan></text>
  <text x="380" y="278" text-anchor="middle" font-size="10" fill="#c0392b" font-style="italic">Cross-shard queries</text>
  <text x="380" y="292" text-anchor="middle" font-size="10" fill="#c0392b" font-style="italic">require app-level code</text>
  <!-- West Shard -->
  <rect x="520" y="110" width="230" height="220" rx="8" fill="#d5f5e3" stroke="#27ae60" stroke-width="2"/>
  <text x="635" y="132" text-anchor="middle" font-weight="bold" font-size="12">West Shard (port 5435)</text>
  <text x="635" y="148" text-anchor="middle" font-size="10" fill="#555">shard3_db — Portland, OR</text>
  <rect x="535" y="160" width="200" height="28" rx="4" fill="#fff" stroke="#6c3483" stroke-width="1"/>
  <text x="635" y="179" text-anchor="middle" font-size="11">warehouses <tspan fill="#888" font-size="9">(replicated)</tspan></text>
  <rect x="535" y="196" width="200" height="28" rx="4" fill="#fff" stroke="#2874a6" stroke-width="1"/>
  <text x="635" y="215" text-anchor="middle" font-size="11">sensors <tspan fill="#888" font-size="9">(301–308)</tspan></text>
  <rect x="535" y="232" width="200" height="28" rx="4" fill="#fff" stroke="#c0392b" stroke-width="1"/>
  <text x="635" y="251" text-anchor="middle" font-size="11">sensor_readings <tspan fill="#888" font-size="9">(~10K)</tspan></text>
</svg>
<p style="font-size: 0.85em; color: #555; margin-top: 0.5em;">Each shard is a fully independent PostgreSQL instance. The <code>warehouses</code> table is replicated to all shards; <code>sensors</code> and <code>sensor_readings</code> are partitioned by region. The shards know nothing about each other.</p>
</div>

## Database Schema

Each shard will hold the same three tables. The schema is created by the lab code (not Docker init) to demonstrate the operational burden of manual sharding --- you must apply identical DDL to every shard yourself.

<div style="display: flex; justify-content: center; gap: 2em; flex-wrap: wrap; margin: 1.5em 0;">
<div style="border: 2px solid #6c3483; border-radius: 6px; min-width: 250px; font-size: 0.85em; background: #f9f5fc;">
<div style="background: #6c3483; color: white; padding: 6px 12px; font-weight: bold; text-align: center;">warehouses <span style="font-weight: normal; font-size: 0.85em;">(3 rows, replicated)</span></div>
<table style="width: 100%; margin: 0; border: none;">
<tr><td style="border: none; padding: 3px 12px;">🔑 <code>warehouse_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">INTEGER PK</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>region</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(20)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>city</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(50)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>state</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(2)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>capacity_sqft</code></td><td style="border: none; padding: 3px 12px; color: #888;">INTEGER</td></tr>
</table>
</div>
<div style="border: 2px solid #2874a6; border-radius: 6px; min-width: 250px; font-size: 0.85em; background: #f0f7fc;">
<div style="background: #2874a6; color: white; padding: 6px 12px; font-weight: bold; text-align: center;">sensors <span style="font-weight: normal; font-size: 0.85em;">(8 per shard)</span></div>
<table style="width: 100%; margin: 0; border: none;">
<tr><td style="border: none; padding: 3px 12px;">🔑 <code>sensor_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">INTEGER PK</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>warehouse_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">INTEGER</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>sensor_type</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(20)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>zone</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(20)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>installed_date</code></td><td style="border: none; padding: 3px 12px; color: #888;">DATE</td></tr>
</table>
</div>
<div style="border: 2px solid #c0392b; border-radius: 6px; min-width: 280px; font-size: 0.85em; background: #fdf2f0;">
<div style="background: #c0392b; color: white; padding: 6px 12px; font-weight: bold; text-align: center;">sensor_readings <span style="font-weight: normal; font-size: 0.85em;">(~10K per shard)</span></div>
<table style="width: 100%; margin: 0; border: none;">
<tr><td style="border: none; padding: 3px 12px;">🔑 <code>reading_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">SERIAL PK</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>sensor_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">INTEGER</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>reading_time</code></td><td style="border: none; padding: 3px 12px; color: #888;">TIMESTAMP</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>value</code></td><td style="border: none; padding: 3px 12px; color: #888;">NUMERIC(10,2)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>unit</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(10)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>alert_level</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(10)</td></tr>
</table>
</div>
</div>

::: {.callout-note title="Key Differences from Lab 1"}
Notice there are **no foreign key constraints** between tables. In a sharded setup, `sensors.warehouse_id` cannot reference `warehouses.warehouse_id` across shards --- PostgreSQL has no way to enforce FKs across independent database instances. The `warehouses` table is a **reference table**, replicated identically on every shard to enable local JOINs.
:::

## Setup and Connection

In [ ]:
import psycopg2
import time
import random
import math
from datetime import datetime, timedelta

# Shard connection parameters (3 shards, no coordinator)
SHARDS = {
    'east': {
        'host': 'localhost', 'port': 5433,
        'dbname': 'shard1_db', 'user': 'shard_user', 'password': 'shard_pass'
    },
    'central': {
        'host': 'localhost', 'port': 5434,
        'dbname': 'shard2_db', 'user': 'shard_user', 'password': 'shard_pass'
    },
    'west': {
        'host': 'localhost', 'port': 5435,
        'dbname': 'shard3_db', 'user': 'shard_user', 'password': 'shard_pass'
    }
}

# Connect to all shards
shard_conns = {}
shard_curs = {}

for region, params in SHARDS.items():
    conn = psycopg2.connect(**params)
    conn.autocommit = True
    shard_conns[region] = conn
    shard_curs[region] = conn.cursor()
    print(f"Connected to shard: {region:8s} (port {params['port']})")

print(f"\nAll 3 shard connections established.")

```text
Connected to shard: east     (port 5433)
Connected to shard: central  (port 5434)
Connected to shard: west     (port 5435)

All 3 shard connections established.
```

## Schema Creation: The Operational Burden

With sharding, there is no central schema manager. You must create **identical tables on every shard** manually --- if you forget one or make a typo, that shard has a different schema.

In [ ]:
# Schema DDL — must be applied to EVERY shard
SCHEMA_DDL = [
    """
    CREATE TABLE IF NOT EXISTS warehouses (
        warehouse_id   INTEGER PRIMARY KEY,
        region         VARCHAR(20)  NOT NULL,
        city           VARCHAR(50)  NOT NULL,
        state          VARCHAR(2)   NOT NULL,
        capacity_sqft  INTEGER      NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS sensors (
        sensor_id      INTEGER PRIMARY KEY,
        warehouse_id   INTEGER      NOT NULL,
        sensor_type    VARCHAR(20)  NOT NULL,
        zone           VARCHAR(20)  NOT NULL,
        installed_date DATE         NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS sensor_readings (
        reading_id     SERIAL PRIMARY KEY,
        sensor_id      INTEGER        NOT NULL,
        reading_time   TIMESTAMP      NOT NULL,
        value          NUMERIC(10,2)  NOT NULL,
        unit           VARCHAR(10)    NOT NULL,
        alert_level    VARCHAR(10)    DEFAULT 'normal'
    )
    """
]

# Apply to ALL shards
for region, cur in shard_curs.items():
    for ddl in SCHEMA_DDL:
        cur.execute(ddl)
    print(f"Schema created on shard: {region}")

print(f"\nApplied {len(SCHEMA_DDL)} CREATE TABLE statements x {len(SHARDS)} shards")
print("= {} total DDL operations. Imagine doing this with 50 shards.".format(
    len(SCHEMA_DDL) * len(SHARDS)
))

```text
Schema created on shard: east
Schema created on shard: central
Schema created on shard: west

Applied 3 CREATE TABLE statements x 3 shards
= 9 total DDL operations. Imagine doing this with 50 shards.
```

## Reference Tables: Replicate Shared Data

The `warehouses` table is small and queried from every shard (e.g., for JOINs). This is a **reference table** --- we replicate it identically on all shards. If a warehouse's data changes, you must update **every shard** manually.

In [ ]:
# Warehouse reference data
WAREHOUSES = [
    (1, 'east',    'Newark',    'NJ', 250000),
    (2, 'central', 'Dallas',    'TX', 300000),
    (3, 'west',    'Portland',  'OR', 200000),
]

# Replicate to ALL shards
for region, cur in shard_curs.items():
    cur.execute("DELETE FROM warehouses")  # idempotent
    for wh in WAREHOUSES:
        cur.execute(
            "INSERT INTO warehouses VALUES (%s, %s, %s, %s, %s)",
            wh
        )
    print(f"Replicated {len(WAREHOUSES)} warehouses to shard: {region}")

print("\nEvery shard has the full warehouses table.")
print("This enables local JOINs between sensors and warehouses.")

```text
Replicated 3 warehouses to shard: east
Replicated 3 warehouses to shard: central
Replicated 3 warehouses to shard: west

Every shard has the full warehouses table.
This enables local JOINs between sensors and warehouses.
```

## Shard Routing: Directing Data to the Right Shard

The core question in sharding: **which shard does this row belong to?** We'll use **list-based routing** --- each region maps to a specific shard. This routing logic lives in your application, not the database.

<div style="text-align: center; margin: 1.5em 0;">
<svg viewBox="0 0 700 160" style="max-width: 700px; font-family: sans-serif;">
  <defs><marker id="arrow-rt" viewBox="0 0 10 10" refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" fill="#2874a6"/></marker></defs>
  <rect x="10" y="45" width="170" height="55" rx="8" fill="#d6eaf8" stroke="#2874a6" stroke-width="2"/>
  <text x="95" y="70" text-anchor="middle" font-weight="bold" font-size="13">Application</text>
  <text x="95" y="87" text-anchor="middle" font-size="11" fill="#555">(Shard Router)</text>
  <line x1="180" y1="60" x2="340" y2="25" stroke="#2874a6" stroke-width="1.5" marker-end="url(#arrow-rt)"/>
  <line x1="180" y1="72" x2="340" y2="72" stroke="#2874a6" stroke-width="1.5" marker-end="url(#arrow-rt)"/>
  <line x1="180" y1="85" x2="340" y2="125" stroke="#2874a6" stroke-width="1.5" marker-end="url(#arrow-rt)"/>
  <text x="260" y="35" text-anchor="middle" font-size="10" fill="#2874a6">region = east</text>
  <text x="270" y="66" text-anchor="middle" font-size="10" fill="#2874a6">region = central</text>
  <text x="260" y="118" text-anchor="middle" font-size="10" fill="#2874a6">region = west</text>
  <rect x="350" y="5" width="180" height="38" rx="6" fill="#d5f5e3" stroke="#27ae60" stroke-width="1.5"/>
  <text x="440" y="28" text-anchor="middle" font-weight="bold" font-size="12">East Shard (5433)</text>
  <rect x="350" y="55" width="180" height="38" rx="6" fill="#d5f5e3" stroke="#27ae60" stroke-width="1.5"/>
  <text x="440" y="78" text-anchor="middle" font-weight="bold" font-size="12">Central Shard (5434)</text>
  <rect x="350" y="108" width="180" height="38" rx="6" fill="#d5f5e3" stroke="#27ae60" stroke-width="1.5"/>
  <text x="440" y="131" text-anchor="middle" font-weight="bold" font-size="12">West Shard (5435)</text>
</svg>
<p style="font-size: 0.85em; color: #555; margin-top: 0.5em;">List-based shard routing: the application maps each region to a specific database instance.</p>
</div>

In [ ]:
# Shard routing function — list-based by region
REGION_TO_SHARD = {
    'east':    'east',
    'central': 'central',
    'west':    'west',
}

def get_shard_cursor(region):
    """Route to the correct shard based on region."""
    shard = REGION_TO_SHARD.get(region)
    if shard is None:
        raise ValueError(f"Unknown region: {region}")
    return shard_curs[shard]

print("List-based shard routing:")
for region, shard in REGION_TO_SHARD.items():
    print(f"  {region:10s} -> shard '{shard}' (port {SHARDS[shard]['port']})")

```text
List-based shard routing:
  east       -> shard 'east' (port 5433)
  central    -> shard 'central' (port 5434)
  west       -> shard 'west' (port 5435)
```

## Data Insertion: Route Sensors and Readings to Shards

Each shard gets its region's sensors and ~10,000 readings (8 sensors x 1,250 readings each). The application is responsible for routing every INSERT to the correct shard.

In [ ]:
# Define sensors for each region
SENSOR_DEFS = {
    'east': [
        (101, 1, 'temperature', 'receiving',  '2024-06-01'),
        (102, 1, 'temperature', 'storage-A',  '2024-06-01'),
        (103, 1, 'temperature', 'storage-B',  '2024-06-15'),
        (104, 1, 'humidity',    'receiving',  '2024-07-01'),
        (105, 1, 'humidity',    'storage-A',  '2024-07-01'),
        (106, 1, 'humidity',    'storage-B',  '2024-07-15'),
        (107, 1, 'temperature', 'cold-store', '2024-08-01'),
        (108, 1, 'humidity',    'cold-store', '2024-08-01'),
    ],
    'central': [
        (201, 2, 'temperature', 'dock-north', '2024-05-01'),
        (202, 2, 'temperature', 'dock-south', '2024-05-01'),
        (203, 2, 'temperature', 'bulk-store', '2024-05-15'),
        (204, 2, 'humidity',    'dock-north', '2024-06-01'),
        (205, 2, 'humidity',    'dock-south', '2024-06-01'),
        (206, 2, 'humidity',    'bulk-store', '2024-06-15'),
        (207, 2, 'temperature', 'hazmat',     '2024-07-01'),
        (208, 2, 'humidity',    'hazmat',     '2024-07-01'),
    ],
    'west': [
        (301, 3, 'temperature', 'inbound',     '2024-08-01'),
        (302, 3, 'temperature', 'outbound',    '2024-08-01'),
        (303, 3, 'temperature', 'main-floor',  '2024-08-15'),
        (304, 3, 'humidity',    'inbound',     '2024-09-01'),
        (305, 3, 'humidity',    'outbound',    '2024-09-01'),
        (306, 3, 'humidity',    'main-floor',  '2024-09-15'),
        (307, 3, 'temperature', 'mezzanine',   '2024-10-01'),
        (308, 3, 'humidity',    'mezzanine',   '2024-10-01'),
    ]
}

# Insert sensors into their respective shards
for region, sensors in SENSOR_DEFS.items():
    cur = get_shard_cursor(region)
    cur.execute("DELETE FROM sensor_readings")  # idempotent
    cur.execute("DELETE FROM sensors")
    for s in sensors:
        cur.execute(
            "INSERT INTO sensors VALUES (%s, %s, %s, %s, %s)",
            s
        )
    print(f"Inserted {len(sensors)} sensors into shard: {region}")

print(f"\nTotal: {sum(len(s) for s in SENSOR_DEFS.values())} sensors across 3 shards")

```text
Inserted 8 sensors into shard: east
Inserted 8 sensors into shard: central
Inserted 8 sensors into shard: west

Total: 24 sensors across 3 shards
```

In [ ]:
# Generate sensor readings — ~10K per shard (30K total)
READINGS_PER_SENSOR = 1250  # ~52 days of hourly data

random.seed(42)  # reproducible

for region, sensors in SENSOR_DEFS.items():
    cur = get_shard_cursor(region)
    count = 0

    for sensor_id, wh_id, stype, zone, inst_date in sensors:
        base_time = datetime(2025, 1, 1)

        values = []
        for i in range(READINGS_PER_SENSOR):
            ts = base_time + timedelta(hours=i)
            hour_angle = 2 * math.pi * (ts.hour / 24.0)

            if stype == 'temperature':
                val = round(72.0 + 5.0 * math.sin(hour_angle) + random.gauss(0, 1.5), 2)
                unit = 'F'
            else:
                val = round(45.0 + 10.0 * math.sin(hour_angle + 0.78) + random.gauss(0, 2.0), 2)
                unit = '%'

            alert = 'normal'
            if stype == 'temperature' and (val > 85 or val < 55):
                alert = 'warning'
            if stype == 'humidity' and (val > 70 or val < 20):
                alert = 'warning'

            values.append((sensor_id, ts, val, unit, alert))

        args_str = ','.join(
            cur.mogrify("(%s,%s,%s,%s,%s)", v).decode() for v in values
        )
        cur.execute(
            f"INSERT INTO sensor_readings (sensor_id, reading_time, value, unit, alert_level) VALUES {args_str}"
        )
        count += READINGS_PER_SENSOR

    print(f"Inserted {count:,} readings into shard: {region}")

total = READINGS_PER_SENSOR * sum(len(s) for s in SENSOR_DEFS.values())
print(f"\nTotal readings across all shards: {total:,}")

```text
Inserted 10,000 readings into shard: east
Inserted 10,000 readings into shard: central
Inserted 10,000 readings into shard: west

Total readings across all shards: 30,000
```

## Single-Shard Queries: Fast and Simple

When your query targets a single region, sharding shines. The query only hits one shard --- less data to scan, no contention with other regions.

In [ ]:
def explain_analyze(cursor, query, params=None):
    """Run EXPLAIN ANALYZE and print the query plan."""
    cursor.execute(f"EXPLAIN ANALYZE {query}", params)
    for row in cursor.fetchall():
        print(row[0])

In [ ]:
# Single-shard query: average temperature in East warehouse
print("SINGLE-SHARD QUERY: East warehouse, Jan-Feb 2025 temperatures")
print("=" * 60)

east_cur = shard_curs['east']

east_cur.execute("""
    SELECT COUNT(*) AS readings,
           ROUND(AVG(value), 2) AS avg_temp,
           ROUND(MIN(value), 2) AS min_temp,
           ROUND(MAX(value), 2) AS max_temp
    FROM sensor_readings sr
    JOIN sensors s ON sr.sensor_id = s.sensor_id
    WHERE s.sensor_type = 'temperature'
      AND sr.reading_time >= '2025-01-15'
      AND sr.reading_time <  '2025-02-15'
""")

row = east_cur.fetchone()
print(f"Readings: {row[0]:,}")
print(f"Avg temp: {row[1]} F")
print(f"Min temp: {row[2]} F")
print(f"Max temp: {row[3]} F")

print("\nThis query only touched the East shard — 10K rows, not 30K.")

```text
SINGLE-SHARD QUERY: East warehouse, Jan-Feb 2025 temperatures
============================================================
Readings: 2,976
Avg temp: 71.98 F
Min temp: 62.93 F
Max temp: 81.96 F

This query only touched the East shard — 10K rows, not 30K.
```

In [ ]:
# EXPLAIN ANALYZE on a simple single-shard query
print("EXPLAIN ANALYZE: Single-shard query (East)")
print("=" * 60)

explain_analyze(east_cur, """
    SELECT AVG(value)
    FROM sensor_readings
    WHERE reading_time >= '2025-01-15'
      AND reading_time <  '2025-02-15'
""")

print("\n** How to read this plan:")
print("   1. 'Seq Scan on sensor_readings' — straightforward scan on this")
print("      shard's local table. No parallelism needed for 10K rows.")
print("   2. rows=5,952 returned, Rows Removed by Filter: 4,048 —")
print("      the shard scanned 5,952 + 4,048 = 10,000 total rows,")
print("      confirming this shard's entire table was read.")
print("   3. Execution Time: 0.89 ms — sub-millisecond because 10K rows is tiny.")

```text
EXPLAIN ANALYZE: Single-shard query (East)
============================================================
Aggregate  (cost=140.66..140.67 rows=1 width=32) (actual time=0.877..0.877 rows=1 loops=1)
  ->  Seq Scan on sensor_readings  (cost=0.00..140.60 rows=22 width=16) (actual time=0.020..0.543 rows=5952 loops=1)
        Filter: ((reading_time >= '2025-01-15 00:00:00'::timestamp without time zone) AND (reading_time < '2025-02-15 00:00:00'::timestamp without time zone))
        Rows Removed by Filter: 4048
Planning Time: 0.026 ms
Execution Time: 0.890 ms

** How to read this plan:
   1. 'Seq Scan on sensor_readings' — straightforward scan on this
      shard's local table. No parallelism needed for 10K rows.
   2. rows=5,952 returned, Rows Removed by Filter: 4,048 —
      the shard scanned 5,952 + 4,048 = 10,000 total rows,
      confirming this shard's entire table was read.
   3. Execution Time: 0.89 ms — sub-millisecond because 10K rows is tiny.
```

::: {.callout-tip title="Single-Shard Advantage"}
The `rows=5952` plus `Rows Removed by Filter: 4048` confirms the shard holds exactly 10,000 rows. Sub-millisecond execution because each shard has only a fraction of the total data.
:::

## Cross-Shard Queries: The Pain Begins

What if you need the hottest sensor across **all** warehouses? There is **no SQL JOIN that spans shards** --- the databases are completely independent. You must query each shard separately and combine results in application code.

In [ ]:
# Application-level cross-shard query
print("CROSS-SHARD QUERY: Hottest reading per region, Jan-Feb 2025")
print("=" * 60)

results = []  # each entry: (region, sensor_id, zone, value, reading_time)

for region, cur in shard_curs.items():
    cur.execute("""
        SELECT sr.sensor_id, s.zone, sr.value, sr.reading_time
        FROM sensor_readings sr
        JOIN sensors s ON sr.sensor_id = s.sensor_id
        WHERE s.sensor_type = 'temperature'
          AND sr.reading_time >= '2025-01-01'
          AND sr.reading_time <  '2025-03-01'
        ORDER BY sr.value DESC
        LIMIT 1
    """)
    row = cur.fetchone()
    #         index:  0       1       2             3        4
    results.append((region, row[0], row[1], float(row[2]), row[3]))
    print(f"  {region:8s}: sensor {row[0]} in {row[1]:12s} -> {row[2]} F at {row[3]}")

# Find the global maximum in Python.
# results is a list of tuples: (region, sensor_id, zone, value, reading_time)
# The lambda picks index 3 (the temperature value) to compare across tuples.
hottest = max(results, key=lambda x: x[3])
print(f"\nGlobal hottest: {hottest[0]} region, sensor {hottest[1]} ({hottest[2]}), {hottest[3]} F")
print("\nThis required 3 separate queries + Python aggregation.")
print("A single-server query would be one SQL statement.")

```text
CROSS-SHARD QUERY: Hottest reading per region, Jan-Feb 2025
============================================================
  east    : sensor 101 in receiving    -> 81.96 F at 2025-01-31 04:00:00
  central : sensor 202 in dock-south   -> 81.28 F at 2025-01-20 03:00:00
  west    : sensor 301 in inbound      -> 81.25 F at 2025-01-27 07:00:00

Global hottest: east region, sensor 101 (receiving), 81.96 F

This required 3 separate queries + Python aggregation.
A single-server query would be one SQL statement.
```

::: {.callout-important title="Cross-Shard Complexity"}
What would be a single `SELECT ... ORDER BY value DESC LIMIT 1` on one server becomes **3 separate queries + application-level comparison**.
:::

## Cross-Shard Aggregation

Business dashboards need aggregates across all regions. With sharding, this means querying every shard and combining results in your application code.

In [ ]:
# Cross-shard aggregation: temperature summary by region
print("CROSS-SHARD AGGREGATION: Temperature summary by region")
print("=" * 70)
print(f"{'Region':10s} {'Readings':>10s} {'Avg Temp':>10s} {'Min':>8s} {'Max':>8s} {'Alerts':>8s}")
print("-" * 70)

global_count = 0
global_sum = 0
global_min = float('inf')
global_max = float('-inf')
global_alerts = 0

for region, cur in shard_curs.items():
    cur.execute("""
        SELECT COUNT(*),
               ROUND(AVG(value), 2),
               ROUND(MIN(value), 2),
               ROUND(MAX(value), 2),
               COUNT(*) FILTER (WHERE alert_level = 'warning')
        FROM sensor_readings
        WHERE unit = 'F'
    """)
    cnt, avg_val, min_val, max_val, alerts = cur.fetchone()

    print(f"{region:10s} {cnt:>10,} {avg_val:>10} {min_val:>8} {max_val:>8} {alerts:>8,}")

    global_count += cnt
    global_sum += float(avg_val) * cnt
    global_min = min(global_min, float(min_val))
    global_max = max(global_max, float(max_val))
    global_alerts += alerts

global_avg = round(global_sum / global_count, 2)
print("-" * 70)
print(f"{'GLOBAL':10s} {global_count:>10,} {global_avg:>10} {global_min:>8} {global_max:>8} {global_alerts:>8,}")

print("\nNotice: MIN and MAX are simple (take the extreme across shards),")
print("but AVG must be computed as a weighted average:")
print("  global_avg = SUM(count * avg) / SUM(count)")
print("Simply averaging the per-shard averages would give the wrong")
print("answer if shards had different row counts.")

```text
CROSS-SHARD AGGREGATION: Temperature summary by region
======================================================================
Region       Readings   Avg Temp      Min      Max   Alerts
----------------------------------------------------------------------
east            5,000      71.99    62.93    81.96        0
central         5,000      72.01    62.84    81.28        0
west            5,000      72.05    62.39    81.25        0
----------------------------------------------------------------------
GLOBAL         15,000      72.02    62.39    81.96        0

Notice: MIN and MAX are simple (take the extreme across shards),
but AVG must be computed as a weighted average:
  global_avg = SUM(count * avg) / SUM(count)
Simply averaging the per-shard averages would give the wrong
answer if shards had different row counts.
```

::: {.callout-note title="Why Weighted Averages?"}
With equal-sized shards, averaging the per-shard averages happens to give the correct answer. But in production, shards often have very different sizes. The safe pattern is always `SUM(count × avg) / SUM(count)`. The comprehensive lab demonstrates this with a worked example.
:::

## Summary: Partitioning vs Sharding

| | **Partitioning** (Lab 1) | **Sharding** (Lab 2) |
|---|---|---|
| **Where** | Single server | Multiple servers |
| **Managed by** | PostgreSQL engine | Your application |
| **JOINs** | Normal SQL | Application-level |
| **Foreign keys** | Fully enforced | Not possible across shards |
| **Schema changes** | One ALTER TABLE | Must apply to every shard |
| **Aggregation** | Normal SQL | Query all shards + combine (weighted!) |
| **Scales** | Up to server limits | Horizontally (add more shards) |

### Key Takeaways

1. **Sharding trades simplicity for scalability** --- you gain horizontal scale but lose SQL convenience
2. **Shard routing is application logic** --- the database doesn't know about other shards
3. **Cross-shard queries are expensive** --- both in latency and code complexity
4. **Aggregations need care** --- use weighted averages, not average-of-averages

### Topics in the Comprehensive Lab

The full version also covers:

- Schema drift demo (missed ALTER TABLE)
- Reference table inconsistency (stale data on one shard)
- Hash-based routing comparison
- Index locality (shard-local indexes)
- FK limitations + application-level enforcement
- dblink coordinator queries

## Cleanup

When you're done, shut down the Docker environment:

```bash
cd sharding-lab
docker compose down -v
```

In [ ]:
# Close all database connections
for region, conn in shard_conns.items():
    shard_curs[region].close()
    conn.close()

print("All connections closed.")

```text
All connections closed.
```